# Sampling Test

## Information

Técnicas de muestreo del módulo `scipy.stats.qmc`:
* Sobol: <--
    * Permite ampliar progresivamente el número de muestras
    * **Se recomienda que el número total de muestras sea una potencia de 2**.
* Latin Hypercube (LHS)
    * Funciona con muestras bajas
    * **NO** ampliar el número de muestras

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# # Librerías auxiliares
# import_dir = Path(settings.get('TVB_IMPORT_DIR') or env_dir).expanduser()
# if not import_dir.is_absolute():
#     import_dir = env_dir / import_dir
# sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import qmc
from scipy.stats import beta, truncnorm, uniform

from sklearn.decomposition import PCA

**Console**

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

## Data

## Basic Sobol Sampling

In [ ]:
N_VARIABLES = 30
M_SAMPLES = 2**10
SEED = 42

In [ ]:
def create_sobol():
    return qmc.Sobol(
        d=N_VARIABLES, scramble=True, seed=SEED,
    )


In [ ]:
sampler = create_sobol()

m = int(np.log2(M_SAMPLES))-1
batch11 = sampler.random_base2(m=m)
batch12 = sampler.random_base2(m=m)
batch1 = np.vstack((batch11, batch12))    

pprint(f'Batch shape: {batch1.shape}, Batch size: {batch1.nbytes / 1024**2:.2f} MB')

In [ ]:
sampler = create_sobol()

# Puedes trabajar sin la restricción de que sea una potencia de 2, **pero no es recomendable**.
batch2 = np.vstack([sampler.random(M_SAMPLES // 4) for _ in range(4)])

pprint(f'Batch shape: {batch2.shape}, Batch size: {batch2.nbytes / 1024**2:.2f} MB')

In [ ]:
# Test para verificar como aplicar el número de muestras en distintos ficheros.

batch3_parts = []
for i in range(4):
    sampler = create_sobol()
    if i != 0: 
        sampler.fast_forward(i * (M_SAMPLES // 4))
    batch3_parts.append(sampler.random(M_SAMPLES // 4))

batch3 = np.vstack(batch3_parts)

In [ ]:
## Proyección PCA de los tres lotes
pca = PCA(n_components=2).fit(batch1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)

for ax, (name, batch) in zip(axes, [('Batch 1', batch1), ('Batch 2', batch2), ('Batch 3', batch3)]):
    projected = pca.transform(batch)
    ax.scatter(projected[:, 0], projected[:, 1], s=1)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])


fig.suptitle('Sobol Sampling PCA Projection')
fig.tight_layout()
plt.show()

## Distributions

In [ ]:
PEAK = 0.2
CONCENTRATION = 5
STD = 0.1
MIN = 0.0
MAX = 1.0

In [ ]:
relative_peak = (PEAK - MIN) / (MAX - MIN)

beta_a = 1 + relative_peak * CONCENTRATION
beta_b = 1 + (1 - relative_peak) * CONCENTRATION

# Truncated normal distribution
trunc_a = (MIN - PEAK) / STD
trunc_b = (MAX - PEAK) / STD

distributions = [
    ("Asymmetric Beta", beta(beta_a, beta_b, loc=MIN, scale=MAX - MIN)),
    ("Truncated Normal", truncnorm(trunc_a, trunc_b, loc=PEAK, scale=STD)),
    ("Uniform", uniform(loc=MIN, scale=MAX - MIN)),
]

In [ ]:
x = np.linspace(0, 1, 1000)

fig, axes = plt.subplots(1, 3, figsize=(10, 7))

for ax, (title, dist) in zip(axes.flat, distributions):
    y = dist.pdf(x)
    ax.plot(x, y)
    ax.fill_between(x, y, alpha=0.2)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("Density")
    ax.set_xlim(0, 1)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()